# 🏀 Basketball Player Detection & Tracking — GOOGLE COLAB VERSION
## Big Vision Internship Assignment | YOLOv11m + ByteTrack

---
## 📌 ALGORITHM SELECTION — INTERVIEW-LEVEL EXPLANATION

### 🔍 Detection: YOLOv11m (You Only Look Once v11, Medium)
**Why chosen:** Single-stage real-time detector. Processes entire image in one pass at 30-100 FPS — essential for tracking 10 simultaneously moving players. Two-stage detectors (Faster R-CNN) run at 5-10 FPS making real-time tracking impossible. DETR (transformer) needs 100k+ images to converge, far beyond our ~5000 image dataset.

**Pre-trained? ✅ YES — COCO dataset (Microsoft, 2017)**
- 118,000 images, 80 classes including `person`
- Source: https://github.com/ultralytics/ultralytics
- We **fine-tune** on basketball data: 90 min instead of days
- Why v11 over v8: 22% fewer params, higher mAP, same API

### 🏃 Tracking: ByteTrack
**Why chosen:** Basketball players constantly screen each other (occlusion). When a player is half-hidden, their detection confidence drops. SORT discards low-confidence detections → loses track → new ID on re-emergence. DeepSORT adds visual re-ID (CNN) but all same-team players wear identical jerseys → CNN confused. ByteTrack's **two-stage matching** uses both high and low confidence detections, keeping tracks alive through occlusion.

**Pre-trained? ✅ YES — Kalman filter motion model (bytetrack.yaml, Ultralytics)**
- No separate training needed — mathematical motion predictor
- Learns each player's velocity and position online per frame
- Reference: ByteTrack paper (Zhang et al., 2022)

| Algorithm | Core Mechanism | Basketball Problem | MOTA |
|-----------|---------------|-------------------|------|
| SORT | Kalman only | Loses ID at player crossings | ~55% |
| DeepSORT | Kalman + appearance CNN | Same jersey = confused | ~65% |
| **ByteTrack ✅** | Kalman + ALL detections | Recovers through occlusion | **77.3%** |

### 🎯 False Detection Fix (Basketball Posts/Hoops)
Basketball posts were being detected as players (similar tall shape).
**Fixes:** (1) Removed referee class — caused post/equipment confusion. (2) Aspect ratio filter: height/width > 5.5 = post shape, rejected. (3) Min area filter: tiny boxes < 0.5% image = noise, rejected. (4) 5 datasets instead of 3.

---
## ⚠️ BEFORE YOU RUN
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Get free Roboflow API key: https://roboflow.com → Settings → Roboflow API
3. Paste key in Cell 2 where it says `YOUR_API_KEY_HERE`
4. Run cells **top to bottom**, one at a time


---
## 📦 CELL 1 — Install Libraries
**Nothing to change. Run once per Colab session.**

In [ ]:
!pip install ultralytics==8.3.40 -q
!pip install roboflow -q
!pip install supervision -q
!pip install lap -q
!pip install gradio -q
!pip install seaborn -q

import os, cv2, shutil, glob, yaml, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image as PILImage
import torch, pandas as pd
warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

print('='*55)
print('  ENVIRONMENT CHECK')
print('='*55)
print(f'  PyTorch  : {torch.__version__}')
print(f'  GPU OK   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU Name : {props.name}')
    print(f'  GPU RAM  : {props.total_memory/1e9:.1f} GB')
else:
    print('  NO GPU — go to Runtime > Change runtime type > T4 GPU')
print('='*55)


---
## 📥 CELL 2 — Download 5 Datasets
**⚠️ CHANGE: Replace `YOUR_API_KEY_HERE` with your Roboflow API key.**
Get it: roboflow.com → Settings → Roboflow API

Downloads ~5000+ labeled basketball images. Classes: player + ball only.

In [ ]:
from roboflow import Roboflow

# ══════════════════════════════════════════════════
# ⚠️  CHANGE THIS — your free Roboflow API key:
API_KEY = 'YOUR_API_KEY_HERE'
# ══════════════════════════════════════════════════

rf = Roboflow(api_key=API_KEY)
DATASETS_CONFIG = [
    ('lokesh-podipireddy-eocdt','basketball-player-detection-6y9yj',1,'/content/dataset1','1616 imgs'),
    ('roboflow-jvuqo','basketball-player-detection-2',1,'/content/dataset2','1398 imgs'),
    ('roboflow-universe-projects','basketball-players-fy4c2',4,'/content/dataset3','600+ imgs'),
    ('sports-detection','basketball-player-detection-8hxbr',1,'/content/dataset4','extra imgs'),
    ('roboflow-jvuqo','basketball',1,'/content/dataset5','full scene'),
]
DATASET_PATHS = []
for ws,proj,ver,loc,desc in DATASETS_CONFIG:
    print(f'Downloading: {desc}')
    try:
        p=rf.workspace(ws).project(proj); d=p.version(ver).download('yolov8',location=loc)
        n=len(glob.glob(f'{loc}/**/*.jpg',recursive=True))
        print(f'  ✅ {n} images → {loc}'); DATASET_PATHS.append(loc)
    except Exception as e: print(f'  ⚠️  Skipped ({e.__class__.__name__})')
for p in ['/content/dataset1','/content/dataset2','/content/dataset3']:
    if p not in DATASET_PATHS and os.path.exists(p): DATASET_PATHS.append(p)
total=sum(len(glob.glob(f'{p}/**/*.jpg',recursive=True)) for p in DATASET_PATHS)
print(f'\n✅ {len(DATASET_PATHS)} datasets | ~{total} images')


---
## 🔍 CELL 3 — Smart Path Detection (Fixes '0 Images' Error)

**Handles both Roboflow folder layouts automatically. Nothing to change.**

In [ ]:
def get_yolo_paths(root, split):
    aliases = {'valid':['valid','val'],'val':['val','valid'],
               'train':['train'],'test':['test']}.get(split,[split])
    for s in aliases:
        img_a = os.path.join(root,'images',s)
        lbl_a = os.path.join(root,'labels',s)
        if os.path.exists(img_a) and (glob.glob(f'{img_a}/*.jpg')+glob.glob(f'{img_a}/*.png')):
            return img_a, lbl_a
        img_b = os.path.join(root,s,'images')
        lbl_b = os.path.join(root,s,'labels')
        if os.path.exists(img_b) and (glob.glob(f'{img_b}/*.jpg')+glob.glob(f'{img_b}/*.png')):
            return img_b, lbl_b
    return None, None
print('Smart path detection ready.')


for i,p in enumerate(DATASET_PATHS,1):
    print(f'\nDataset {i}: {p}')
    if not os.path.exists(p): print('  NOT FOUND'); continue
    yf=os.path.join(p,'data.yaml')
    if os.path.exists(yf):
        with open(yf) as f: info=yaml.safe_load(f)
        print(f'  Classes: {info.get("names",[])}') 
    t=0
    for sp in ['train','valid','test']:
        d,_=get_yolo_paths(p,sp)
        if d:
            n=len(glob.glob(f'{d}/*.jpg')+glob.glob(f'{d}/*.png'))
            print(f'  {sp}: {n} → {d}'); t+=n
    print(f'  TOTAL: {t}')

---
## 🧹 CELL 4 — Quality Filter
**Removes blurry/dark/corrupted images. Nothing to change.**

In [ ]:
def quality_filter(dataset_path):
    removed, kept = 0, 0
    for split in ['train','valid','val','test']:
        img_dir, lbl_dir = get_yolo_paths(dataset_path, split)
        if not img_dir: continue
        images = glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png')
        for ip in images:
            reason = None
            try: PILImage.open(ip).verify()
            except: reason = 'corrupted'
            if reason is None:
                gray = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
                if gray is None: reason = 'unreadable'
                else:
                    if cv2.Laplacian(gray,cv2.CV_64F).var() < 80: reason = 'blurry'
                    else:
                        bgr = cv2.imread(ip)
                        if bgr is not None:
                            b = cv2.cvtColor(bgr,cv2.COLOR_BGR2HSV)[:,:,2].mean()
                            if b < 25: reason='dark'
                            elif b > 235: reason='bright'
            if reason:
                os.remove(ip)
                if lbl_dir:
                    lp = os.path.join(lbl_dir, Path(ip).stem+'.txt')
                    if os.path.exists(lp): os.remove(lp)
                removed += 1
            else: kept += 1
    return kept, removed

print('Quality filtering...')
for i,p in enumerate(DATASET_PATHS,1):
    if os.path.exists(p):
        k,r=quality_filter(p)
        print(f'  Dataset {i}: kept={k}, removed={r}')
print('Done!')

---
## ⚙️ CELL 5 — Preprocess (Resize 640×640)
**Nothing to change.**

In [ ]:
def preprocess_images(dataset_path):
    resized, converted, errors = 0, 0, 0
    for split in ['train','valid','val','test']:
        img_dir, _ = get_yolo_paths(dataset_path, split)
        if not img_dir: continue
        for ip in glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png'):
            try:
                img = cv2.imread(ip)
                if img is None: errors+=1; continue
                if len(img.shape)==2:
                    img=cv2.cvtColor(img,cv2.COLOR_GRAY2BGR); converted+=1
                if (img.shape[1],img.shape[0])!=(640,640):
                    img=cv2.resize(img,(640,640),interpolation=cv2.INTER_LINEAR); resized+=1
                cv2.imwrite(ip,img,[cv2.IMWRITE_JPEG_QUALITY,95])
            except: errors+=1
    return resized, converted, errors

print('Preprocessing...')
for i,p in enumerate(DATASET_PATHS,1):
    if os.path.exists(p):
        r,c,e=preprocess_images(p)
        print(f'  Dataset {i}: resized={r}, converted={c}, errors={e}')
print('Done!')

---
## 🔀 CELL 6 — Merge Datasets (player + ball only)

**Key accuracy fix:** referee class removed. Aspect ratio + area filters prevent post/hoop false detections.

**Nothing to change.**

In [ ]:
TARGET_CLASSES = {0:'player', 1:'ball'}

def get_cls_map(yaml_path):
    if not os.path.exists(yaml_path): return {0:'player'}
    with open(yaml_path) as f: info = yaml.safe_load(f)
    return {i:n.lower() for i,n in enumerate(info.get('names',['player']))}

def remap_cls(orig_id, src_map):
    name = src_map.get(orig_id,'player').lower()
    if any(k in name for k in ['player','person','athlete']): return 0
    if any(k in name for k in ['ball','basketball']): return 1
    return None

def is_valid_box(cx, cy, bw, bh, cls_id):
    if bw*bh < 0.005: return False
    if cls_id==0:
        if bh>0 and bw/max(bh,1e-5) > 4.5: return False
        if bw>0 and bh/max(bw,1e-5) > 5.5: return False
    return True

def merge_datasets(src_dirs, target_dir, yaml_path_key='path'):
    os.makedirs(target_dir, exist_ok=True)
    for split in ['train','valid','test']:
        os.makedirs(f'{target_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{target_dir}/labels/{split}', exist_ok=True)
    counts={'train':0,'valid':0,'test':0}; skipped=0
    for di,src in enumerate(src_dirs):
        if not os.path.exists(src): continue
        src_map = get_cls_map(os.path.join(src,'data.yaml'))
        for tgt_split in ['train','valid','test']:
            img_dir,lbl_dir = get_yolo_paths(src,tgt_split)
            if not img_dir: continue
            images = glob.glob(f'{img_dir}/*.jpg')+glob.glob(f'{img_dir}/*.png')
            for ip in images:
                stem=Path(ip).stem; ext=Path(ip).suffix; nn=f'ds{di}_{stem}'
                shutil.copy(ip,f'{target_dir}/images/{tgt_split}/{nn}{ext}')
                sl=os.path.join(lbl_dir or '',stem+'.txt')
                dl=f'{target_dir}/labels/{tgt_split}/{nn}.txt'
                if os.path.exists(sl):
                    lines=[]
                    with open(sl) as f:
                        for line in f:
                            parts=line.strip().split()
                            if len(parts)!=5: continue
                            nc=remap_cls(int(parts[0]),src_map)
                            if nc is None: skipped+=1; continue
                            cx,cy,bw,bh=map(float,parts[1:])
                            if not is_valid_box(cx,cy,bw,bh,nc): skipped+=1; continue
                            lines.append(f'{nc} {chr(32).join(parts[1:])}\n')
                    with open(dl,'w') as f: f.writelines(lines)
                else: open(dl,'w').close()
                counts[tgt_split]+=1
    return counts, skipped

print('Merging datasets...')
counts,skipped=merge_datasets(
    DATASET_PATHS,'/content/merged_dataset','path')
data_yaml={'path':'/content/merged_dataset','train':'images/train','val':'images/valid','test':'images/test','nc':2,'names':['player','ball']}
with open('/content/merged_dataset/data.yaml','w') as f: yaml.dump(data_yaml,f,default_flow_style=False)
print(f'  Skipped {skipped} false-detection boxes')
for sp,n in counts.items(): print(f'  {sp}: {n}')
print(f'  TOTAL: {sum(counts.values())}')
print('\nMerged! Classes: player(0), ball(1)')

---
## ✅ CELL 7 — Verify + Class Distribution Chart
**Nothing to change.**

In [ ]:
errors,total,vc=[],0,0
for split in ['train','valid','test']:
    ldir=f'/content/merged_dataset/labels/{split}'
    if not os.path.exists(ldir): continue
    for lp in glob.glob(f'{ldir}/*.txt'):
        total+=1; ok=True
        with open(lp) as f:
            for ln,line in enumerate(f):
                parts=line.strip().split()
                if not parts: continue
                if len(parts)!=5: errors.append(f'Bad:{Path(lp).name}:{ln}'); ok=False
                else:
                    try:
                        cid=int(parts[0]); vals=list(map(float,parts[1:]))
                        if not all(0<=v<=1 for v in vals): errors.append(f'Range:{ln}'); ok=False
                    except: errors.append(f'Non-num:{ln}'); ok=False
        if ok: vc+=1
print(f'Labels: total={total}, valid={vc}, errors={len(errors)}')
counts={s:{0:0,1:0} for s in ['train','valid','test']}
for split in ['train','valid','test']:
    ldir=f'/content/merged_dataset/labels/{split}'
    if not os.path.exists(ldir): continue
    for lp in glob.glob(f'{ldir}/*.txt'):
        with open(lp) as f:
            for line in f:
                p=line.strip().split()
                if len(p)==5:
                    c=int(p[0])
                    if c in counts[split]: counts[split][c]+=1
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,split in zip(axes,['train','valid','test']):
    vals=[counts[split][k] for k in [0,1]]
    bars=ax.bar(['player','ball'],vals,color=['#2196F3','#FF9800'],edgecolor='black',lw=0.5)
    for bar in bars:
        h=bar.get_height()
        if h>0: ax.text(bar.get_x()+bar.get_width()/2,h+5,f'{int(h):,}',ha='center',va='bottom',fontsize=10,fontweight='bold')
    ax.set_title(f'{split} — {sum(vals):,} annotations')
plt.suptitle('Class Distribution — Merged Dataset',fontsize=13); plt.tight_layout()
plt.savefig('/content/class_distribution.png',dpi=120,bbox_inches='tight'); plt.show()
print('Chart saved!')

---
## 🖼️ CELL 8 — Visualize Annotated Samples
**Visual check before training. Nothing to change.**

In [ ]:
imgs=(glob.glob('/content/merged_dataset/images/train/*.jpg')+glob.glob('/content/merged_dataset/images/train/*.png'))
if not imgs: print('No images. Check Cell 6.')
else:
    random.shuffle(imgs); samples=imgs[:9]
    CLS_C={0:(0.2,0.6,1.0),1:(1.0,0.6,0.0)}; CLS_N={0:'player',1:'ball'}
    fig,axes=plt.subplots(3,3,figsize=(18,16)); axes=axes.flatten()
    for idx,ip in enumerate(samples):
        img=cv2.imread(ip)
        if img is None: axes[idx].axis('off'); continue
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB); h,w=img.shape[:2]
        axes[idx].imshow(img)
        lp=ip.replace('/images/','/labels/').replace('.jpg','.txt').replace('.png','.txt'); n=0
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    parts=line.strip().split()
                    if len(parts)!=5: continue
                    c=int(parts[0]); cx,cy,bw,bh=map(float,parts[1:])
                    x1=(cx-bw/2)*w; y1=(cy-bh/2)*h; color=CLS_C.get(c,(1,1,1))
                    rect=mpatches.Rectangle((x1,y1),bw*w,bh*h,lw=2.5,edgecolor=color,facecolor='none')
                    axes[idx].add_patch(rect)
                    axes[idx].text(x1,y1-4,CLS_N.get(c,'?'),fontsize=8,color='white',bbox=dict(boxstyle='round,pad=0.2',facecolor=color,alpha=0.9)); n+=1
        axes[idx].axis('off'); axes[idx].set_title(f'{Path(ip).name[:28]} ({n} boxes)',fontsize=7)
    for idx in range(len(samples),9): axes[idx].axis('off')
    handles=[mpatches.Patch(color=(0.2,0.6,1.0),label='Player'),mpatches.Patch(color=(1.0,0.6,0.0),label='Ball')]
    fig.legend(handles=handles,loc='lower center',ncol=2,fontsize=12,bbox_to_anchor=(0.5,-0.01))
    plt.suptitle('Training Samples — Verify before training',fontsize=13); plt.tight_layout()
    plt.savefig('/content/sample_annotations.png',dpi=120,bbox_inches='tight'); plt.show()
    print('Samples saved!')

---
## 🚀 CELL 9 — Train YOLOv11m ⭐

**Pre-trained:** COCO (Microsoft, 118k images). Fine-tuned here on basketball data.

**Where used:** Cell 14 (detection test), Cell 15 (ByteTrack backbone).

**⚠️ Optional:** `epochs=150` more accuracy. `batch=8` if CUDA OOM.

In [ ]:
from ultralytics import YOLO
MERGED_DIR='/content/merged_dataset'; RUNS_DIR='/content/runs'
OUTPUTS_DIR='/content'; WEIGHTS_DIR='/content'
BATCH_SIZE=16; DEVICE=0; WORKERS=2  # change batch to 8 if CUDA OOM
model=YOLO('yolo11m.pt')
print('Training YOLOv11m (COCO pretrained, fine-tuning)...')
results=model.train(
    data=os.path.join(MERGED_DIR,'data.yaml'),
    epochs=100, imgsz=640, batch=BATCH_SIZE, device=DEVICE, workers=WORKERS,
    optimizer='AdamW', lr0=0.001, lrf=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5.0,
    translate=0.1, scale=0.5, fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
    pretrained=True, patience=20, conf=0.001, iou=0.6,
    save=True, save_period=10, plots=True, val=True,
    project=RUNS_DIR, name='yolov11m_basketball', exist_ok=True, verbose=True)
BEST_MODEL_PATH=os.path.join(RUNS_DIR,'yolov11m_basketball','weights','best.pt')
print(f'\nTraining done! Best model: {BEST_MODEL_PATH}')

---
## 📈 CELL 10 — Show Training Curves
**Nothing to change.**

In [ ]:
from IPython.display import Image, display
RUN_DIR='/content/runs/yolov11m_basketball/'
for pf,title in [('results.png','Loss + mAP Curves'),('confusion_matrix.png','Confusion Matrix'),
                  ('PR_curve.png','PR Curve'),('F1_curve.png','F1 Curve'),
                  ('val_batch0_pred.jpg','Sample Val Predictions')]:
    fp=RUN_DIR+pf
    if os.path.exists(fp): print(f'\n{title}:'); display(Image(filename=fp,width=800))

---
## 📊 CELL 11 — Full Evaluation + Advanced Graphs

**Computes:** mAP@50, mAP@50-95, Precision, Recall, F1, per-class breakdown.

**Graphs:** learning curves, LR schedule, per-class mAP bars, overall metrics bar.

**Nothing to change.**

In [ ]:
from ultralytics import YOLO
best_model=YOLO(BEST_MODEL_PATH)
metrics=best_model.val(data=os.path.join(MERGED_DIR,'data.yaml'),
    split='test',conf=0.001,iou=0.6,plots=True,save_json=True,verbose=False)
p=metrics.box.mp; r=metrics.box.mr; f1=2*p*r/(p+r+1e-9)
print('='*55)
print('  DETECTION EVALUATION RESULTS')
print('='*55)
print(f'  mAP@50     : {metrics.box.map50:.4f}  (target > 0.85)')
print(f'  mAP@50-95  : {metrics.box.map:.4f}  (target > 0.65)')
print(f'  Precision  : {p:.4f}')
print(f'  Recall     : {r:.4f}')
print(f'  F1 Score   : {f1:.4f}')
print('-'*55)
per_class_map={}
for i,name in enumerate(['player','ball']):
    try:
        v=metrics.box.maps[i]; per_class_map[name]=v
        print(f'  {name:10s}: mAP50={v:.4f}  {chr(9608)*int(v*25)}')
    except: pass
print('='*55)
EVAL_RESULTS={'mAP50':metrics.box.map50,'mAP50_95':metrics.box.map,'precision':p,'recall':r,'f1':f1}
csv_path=os.path.join(RUNS_DIR,'yolov11m_basketball','results.csv')
if os.path.exists(csv_path):
    df_t=pd.read_csv(csv_path); df_t.columns=[c.strip() for c in df_t.columns]
    fig,axes=plt.subplots(2,3,figsize=(18,10))
    cfgs=[(axes[0,0],'train/box_loss','Box Loss Train','#e74c3c'),
          (axes[0,1],'train/cls_loss','Class Loss Train','#3498db'),
          (axes[0,2],'val/box_loss','Val Box Loss','#e67e22'),
          (axes[1,0],'metrics/mAP50(B)','mAP@50','#2ecc71'),
          (axes[1,1],'metrics/precision(B)','Precision','#9b59b6'),
          (axes[1,2],'metrics/recall(B)','Recall','#1abc9c')]
    for ax,col,title,color in cfgs:
        if col in df_t.columns:
            ax.plot(df_t.index,df_t[col],color=color,lw=2)
            ax.fill_between(df_t.index,df_t[col],alpha=0.15,color=color)
            ax.set_title(title,fontsize=12,fontweight='bold'); ax.set_xlabel('Epoch'); ax.grid(True,alpha=0.3)
    plt.suptitle('Training Learning Curves',fontsize=14); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'learning_curves.png'),dpi=130,bbox_inches='tight'); plt.show()
    if 'lr/pg0' in df_t.columns:
        plt.figure(figsize=(10,4))
        plt.plot(df_t.index,df_t['lr/pg0'],color='#e74c3c',lw=2,label='LR Group 0')
        if 'lr/pg1' in df_t.columns: plt.plot(df_t.index,df_t['lr/pg1'],color='#3498db',lw=2,label='LR Group 1')
        plt.xlabel('Epoch'); plt.ylabel('Learning Rate'); plt.title('LR Schedule (Warmup+Cosine Decay)',fontsize=13)
        plt.legend(); plt.grid(True,alpha=0.3); plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR,'lr_schedule.png'),dpi=120); plt.show()
if per_class_map:
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    bars=axes[0].bar(list(per_class_map.keys()),list(per_class_map.values()),color=['#2196F3','#FF9800'],edgecolor='black')
    for bar in bars:
        h=bar.get_height(); axes[0].text(bar.get_x()+bar.get_width()/2,h+0.01,f'{h:.3f}',ha='center',va='bottom',fontsize=12,fontweight='bold')
    axes[0].set_ylim(0,1.1); axes[0].axhline(y=0.85,color='red',ls='--',lw=1.5,label='Target 0.85')
    axes[0].set_title('Per-Class mAP@50',fontsize=13); axes[0].legend()
    mn=['mAP@50','mAP@50-95','Precision','Recall','F1']; mv=[metrics.box.map50,metrics.box.map,p,r,f1]
    cols=['#2196F3','#4CAF50','#9C27B0','#FF5722','#607D8B']
    bars2=axes[1].barh(mn,mv,color=cols,edgecolor='black')
    for bar in bars2:
        w=bar.get_width(); axes[1].text(w+0.005,bar.get_y()+bar.get_height()/2,f'{w:.4f}',va='center',fontsize=11,fontweight='bold')
    axes[1].set_xlim(0,1.15); axes[1].axvline(x=0.85,color='red',ls='--',lw=1.5,label='Target 0.85')
    axes[1].set_title('Overall Metrics',fontsize=13); axes[1].legend()
    plt.suptitle('Detection Accuracy Analysis',fontsize=14); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'detection_metrics.png'),dpi=130,bbox_inches='tight'); plt.show()
print('Evaluation complete!')


---
## 🎬 CELL 12 — Download Test Video

**⚠️ If auto-download fails:** upload any basketball video via Colab Files panel and set `INPUT_VIDEO='/content/your_file.mp4'`

In [ ]:
import urllib.request
VIDEO_URLS=['https://videos.pexels.com/video-files/8425786/8425786-hd_1920_1080_25fps.mp4',
            'https://videos.pexels.com/video-files/6077784/6077784-hd_1280_720_25fps.mp4',
            'https://videos.pexels.com/video-files/5473157/5473157-hd_1280_720_24fps.mp4']
INPUT_VIDEO=None; OUTPUT_VIDEO='/content/basketball_tracked.mp4'
for url in VIDEO_URLS:
    sp='/content/basketball_game.mp4'
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0 Chrome/120.0'})
        with urllib.request.urlopen(req,timeout=60) as resp:
            with open(sp,'wb') as f: f.write(resp.read())
        mb=os.path.getsize(sp)/1e6
        if mb>1.0: INPUT_VIDEO=sp; print(f'Downloaded {mb:.1f} MB'); break
    except Exception as e: print(f'Failed: {e}')
if INPUT_VIDEO is None:
    print('Auto-download failed. Upload manually and set:')
    INPUT_VIDEO='/content/basketball_game.mp4'  # CHANGE THIS if uploading manually
else:
    cap=cv2.VideoCapture(INPUT_VIDEO); fps=cap.get(cv2.CAP_PROP_FPS)
    w=int(cap.get(3)); h=int(cap.get(4)); fr=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    print(f'  {w}x{h} @ {fps:.0f}fps | {fr/fps:.1f}s ({fr} frames)')

---
## 🎞️ CELL 13 — Extract Frames
**Nothing to change.**

In [ ]:
os.makedirs('/content/extracted_frames',exist_ok=True)
cap=cv2.VideoCapture(INPUT_VIDEO); vfps=cap.get(cv2.CAP_PROP_FPS)
tf=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); interval=max(1,int(vfps/5))
saved=0; skipped=0
for fi in range(tf):
    ret,frame=cap.read()
    if not ret: break
    if fi%interval!=0: continue
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    if cv2.Laplacian(gray,cv2.CV_64F).var()<50: skipped+=1; continue
    cv2.imwrite(f'/content/extracted_frames/frame_{saved:05d}.jpg',frame,[cv2.IMWRITE_JPEG_QUALITY,95])
    saved+=1
cap.release(); print(f'Extracted {saved} frames (skipped {skipped} blurry)')
flist=sorted(glob.glob('/content/extracted_frames/*.jpg'))[:4]
if flist:
    fig,axes=plt.subplots(1,4,figsize=(20,5))
    for ax,fp in zip(axes,flist):
        img=cv2.imread(fp); ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
        ax.axis('off'); ax.set_title(Path(fp).name,fontsize=8)
    plt.suptitle('Sample Frames',fontsize=12); plt.tight_layout()
    plt.savefig('/content/sample_frames.png',dpi=100); plt.show()

---
## 🔍 CELL 14 — Detection Test on Sample Frames
**Visual check before full tracking. Nothing to change.**

In [ ]:
from ultralytics import YOLO
best_model=YOLO(BEST_MODEL_PATH); sframes=sorted(glob.glob('/content/extracted_frames/*.jpg'))[:6]
if not sframes: print('Run Cell 13 first.')
else:
    CLS_C={(0):(255,120,0),(1):(0,200,255)}; CLS_N={0:'Player',1:'Ball'}
    fig,axes=plt.subplots(2,3,figsize=(20,12)); axes=axes.flatten()
    for idx,fp in enumerate(sframes):
        frame=cv2.imread(fp); result=best_model.predict(frame,conf=0.40,iou=0.5,verbose=False)[0]
        for box in result.boxes:
            x1,y1,x2,y2=map(int,box.xyxy[0]); cid=int(box.cls[0]); conf=float(box.conf[0])
            color=CLS_C.get(cid,(255,255,255)); label=f'{CLS_N.get(cid,"?")} {conf:.2f}'
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            (tw,th),_=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
            cv2.rectangle(frame,(x1,y1-th-6),(x1+tw+4,y1),color,-1)
            cv2.putText(frame,label,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
        axes[idx].imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)); axes[idx].axis('off')
        axes[idx].set_title(f'Frame {idx+1} — {len(result.boxes)} detections',fontsize=10)
    for idx in range(len(sframes),6): axes[idx].axis('off')
    plt.suptitle('Detection Test — YOLOv11m',fontsize=13); plt.tight_layout()
    plt.savefig('/content/detection_test.png',dpi=120,bbox_inches='tight'); plt.show()

---
## 🏃 CELL 15 — Full Tracking with ByteTrack ⭐

**Algorithm: ByteTrack** — called via `model.track(tracker='bytetrack.yaml', persist=True)` every frame.

**Why ByteTrack:** Two-stage matching uses low-confidence detections to keep partially-occluded players tracked. SORT and DeepSORT both fail at basketball occlusions. ByteTrack: MOTA=77.3%, 558 ID switches vs 1000+ for SORT.

**Nothing to change.**

In [ ]:
import supervision as sv
from ultralytics import YOLO

best_model  = YOLO(BEST_MODEL_PATH)
CLASS_NAMES = {0:'Player', 1:'Ball'}

if not os.path.exists(INPUT_VIDEO):
    print(f'Video not found: {INPUT_VIDEO}')
else:
    video_info = sv.VideoInfo.from_video_path(INPUT_VIDEO)
    print(f'Processing: {video_info.width}x{video_info.height} @ {video_info.fps}fps | {video_info.total_frames} frames')

    box_ann   = sv.BoxAnnotator(thickness=2)
    lbl_ann   = sv.LabelAnnotator(text_scale=0.5,text_thickness=1,text_padding=3)
    trace_ann = sv.TraceAnnotator(thickness=2,trace_length=50)
    tracking_data = []

    with sv.VideoSink(OUTPUT_VIDEO, video_info) as sink:
        for fi, frame in enumerate(sv.get_video_frames_generator(INPUT_VIDEO)):
            # ByteTrack detection + tracking
            result = best_model.track(
                frame, tracker='bytetrack.yaml',
                persist=True, conf=0.40, iou=0.5, verbose=False)[0]
            dets = sv.Detections.from_ultralytics(result)
            # Post-filter: remove post/hoop shaped boxes
            if len(dets)>0:
                keep=[]
                for i in range(len(dets)):
                    x1,y1,x2,y2=dets.xyxy[i]; bw=x2-x1; bh=y2-y1
                    if int(dets.class_id[i])==0:
                        if bh>0 and bh/max(bw,1)>5.5: continue
                        if bw>0 and bw/max(bh,1)>4.5: continue
                    keep.append(i)
                if keep: dets=dets[keep]
            labels=[]
            for i in range(len(dets)):
                tid=int(dets.tracker_id[i]) if dets.tracker_id is not None else -1
                cls=int(dets.class_id[i]); conf=float(dets.confidence[i])
                labels.append(f'#{tid} {CLASS_NAMES.get(cls,"?")} {conf:.2f}')
                if dets.tracker_id is not None:
                    tracking_data.append({'frame':fi,'track_id':tid,
                        'class_id':cls,'conf':conf,'xyxy':dets.xyxy[i].tolist()})
            ann=frame.copy()
            ann=trace_ann.annotate(ann,dets)
            ann=box_ann.annotate(ann,dets)
            ann=lbl_ann.annotate(ann,dets,labels)
            np_=sum(1 for c in dets.class_id if c==0)
            cv2.rectangle(ann,(0,0),(520,32),(0,0,0),-1)
            cv2.putText(ann,f'Frame:{fi:4d} | Players:{np_} | ByteTrack+YOLOv11m',
                (6,22),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),1)
            sink.write_frame(ann)
            if fi%60==0:
                pct=fi/video_info.total_frames*100
                print(f'  {fi}/{video_info.total_frames} ({pct:.0f}%) det={len(dets)}')

    print(f'\nTracking complete! Output: {OUTPUT_VIDEO}')
    print(f'Total tracking entries: {len(tracking_data)}')

# Auto-display in notebook
from base64 import b64encode
from IPython.display import HTML
if os.path.exists(OUTPUT_VIDEO) and os.path.getsize(OUTPUT_VIDEO)>0:
    with open(OUTPUT_VIDEO,'rb') as f: vid64=b64encode(f.read()).decode()
    display(HTML(f'<video width="800" controls autoplay loop><source src="data:video/mp4;base64,{vid64}" type="video/mp4"></video>'))

---
## 📊 CELL 16 — Tracking Analytics Dashboard
**6 charts: track length distribution, confidence by class, detections/frame, players/frame, CDF, rolling confidence. Nothing to change.**

In [ ]:
if 'tracking_data' not in dir() or not tracking_data:
    print('Run tracking cell first.')
else:
    df=pd.DataFrame(tracking_data)
    player_df=df[df['class_id']==0]
    track_lens=df.groupby('track_id').size()
    short=(track_lens<5).sum()
    long_pct=(track_lens>=20).sum()/max(len(track_lens),1)*100
    mota_approx=max(0.0,1.0-(short/max(len(df),1)))

    print('='*55)
    print('  TRACKING STATISTICS (ByteTrack)')
    print('='*55)
    print(f'  Detections  : {len(df):,}')
    print(f'  Track IDs   : {df["track_id"].nunique()}')
    print(f'  Avg conf    : {df["conf"].mean():.4f}')
    print(f'  Avg track   : {track_lens.mean():.1f} frames')
    print(f'  Short (<5f) : {short}')
    print(f'  Long (>=20f): {long_pct:.1f}%')
    print(f'  Approx MOTA : {mota_approx:.4f}')
    print('='*55)

    fig, axes = plt.subplots(2,3,figsize=(18,11))
    axes[0,0].hist(track_lens.values,bins=30,color='#2196F3',edgecolor='black',lw=0.5)
    axes[0,0].axvline(x=5,color='red',ls='--',lw=2,label='<5=ID switch')
    axes[0,0].axvline(x=20,color='green',ls='--',lw=2,label='>=20=stable')
    axes[0,0].set_title('Track Length Distribution',fontsize=11,fontweight='bold')
    axes[0,0].legend()
    for cid,nm,col in [(0,'Player','#2196F3'),(1,'Ball','#FF9800')]:
        sub=df[df['class_id']==cid]
        if len(sub)>0:
            axes[0,1].hist(sub['conf'].values,bins=25,alpha=0.65,label=nm,color=col,edgecolor='black',lw=0.3)
    axes[0,1].set_title('Confidence by Class',fontsize=11,fontweight='bold'); axes[0,1].legend()
    dpf=df.groupby('frame').size()
    axes[0,2].plot(dpf.index,dpf.values,color='#9C27B0',lw=1.2)
    axes[0,2].fill_between(dpf.index,dpf.values,alpha=0.2,color='#9C27B0')
    axes[0,2].set_title('Detections Per Frame',fontsize=11,fontweight='bold')
    ppf=player_df.groupby('frame').size()
    axes[1,0].plot(ppf.index,ppf.values,color='#2196F3',lw=1.5)
    axes[1,0].axhline(y=ppf.mean(),color='red',ls='--',lw=1.5,label=f'Mean={ppf.mean():.1f}')
    axes[1,0].set_title('Players Per Frame',fontsize=11,fontweight='bold'); axes[1,0].legend()
    sl=np.sort(track_lens.values); cdf=np.arange(1,len(sl)+1)/len(sl)
    axes[1,1].plot(sl,cdf,color='#4CAF50',lw=2); axes[1,1].axvline(x=20,color='red',ls='--',lw=1.5)
    axes[1,1].set_title('Track Length CDF',fontsize=11,fontweight='bold')
    rc=df.groupby('frame')['conf'].mean().rolling(30).mean()
    axes[1,2].plot(rc.index,rc.values,color='#FF5722',lw=1.5)
    axes[1,2].set_title('Rolling Avg Confidence (30f)',fontsize=11,fontweight='bold'); axes[1,2].set_ylim(0,1)
    plt.suptitle('ByteTrack Analytics Dashboard',fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'tracking_analytics.png'),dpi=130,bbox_inches='tight')
    plt.show()
    print('Tracking analytics saved!')


---
## 🔥 CELL 17 — Spatial Analytics (Heatmaps + Zone Analysis)
**Player heatmap, ball heatmap, zone occupancy, density grid, trajectories, temporal activity. Nothing to change.**

In [ ]:
if 'tracking_data' not in dir() or not tracking_data:
    print('Run tracking cell first.')
else:
    cap=cv2.VideoCapture(INPUT_VIDEO)
    W=int(cap.get(3)); H=int(cap.get(4)); ret,bg=cap.read(); cap.release()
    df=pd.DataFrame(tracking_data)
    player_df=df[df['class_id']==0]; ball_df=df[df['class_id']==1]

    def build_hm(df_sub):
        hm=np.zeros((H,W),dtype=np.float32)
        for _,row in df_sub.iterrows():
            x1,y1,x2,y2=row['xyxy']; cx=int((x1+x2)/2); cy=int((y1+y2)/2)
            if 0<=cy<H and 0<=cx<W: hm[cy,cx]+=1
        hm=cv2.GaussianBlur(hm,(61,61),0)
        if hm.max()>0: hm=(hm/hm.max()*255).astype(np.uint8)
        return hm

    p_hm=build_hm(player_df); b_hm=build_hm(ball_df) if len(ball_df)>0 else np.zeros((H,W),np.uint8)
    fig=plt.figure(figsize=(20,14))
    ax1=fig.add_subplot(2,3,1)
    cp=cv2.applyColorMap(p_hm,cv2.COLORMAP_JET)
    ov=cv2.addWeighted(bg,0.35,cp,0.65,0) if bg is not None else cp
    cv2.imwrite(os.path.join(OUTPUTS_DIR,'player_heatmap.jpg'),ov)
    ax1.imshow(cv2.cvtColor(ov,cv2.COLOR_BGR2RGB))
    ax1.set_title('Player Movement Heatmap',fontsize=12,fontweight='bold'); ax1.axis('off')
    ax2=fig.add_subplot(2,3,2)
    cb=cv2.applyColorMap(b_hm,cv2.COLORMAP_HOT)
    ax2.imshow(cv2.cvtColor(cb,cv2.COLOR_BGR2RGB))
    ax2.set_title('Ball Movement Heatmap',fontsize=12,fontweight='bold'); ax2.axis('off')
    ax3=fig.add_subplot(2,3,3)
    zones={'Left Paint':0,'Left Mid':0,'Left 3pt':0,'Right Paint':0,'Right Mid':0,'Right 3pt':0}
    for _,row in player_df.iterrows():
        x1,y1,x2,y2=row['xyxy']; cx=(x1+x2)/2/W; cy=(y1+y2)/2/H
        side='Left' if cx<0.5 else 'Right'
        if cy<0.35 or cy>0.65: zones[f'{side} Paint']+=1
        elif 0.35<=cy<=0.65 and ((side=='Left' and cx<0.33) or (side=='Right' and cx>0.67)): zones[f'{side} Mid']+=1
        else: zones[f'{side} 3pt']+=1
    zc=['#FF6B6B','#FFA07A','#FFD700','#90EE90','#87CEEB','#DDA0DD']
    ax3.pie(zones.values(),labels=zones.keys(),colors=zc,autopct='%1.1f%%',startangle=90)
    ax3.set_title('Zone Occupancy',fontsize=12,fontweight='bold')
    ax4=fig.add_subplot(2,3,4)
    gw,gh=10,6; density=np.zeros((gh,gw))
    for _,row in player_df.iterrows():
        x1,y1,x2,y2=row['xyxy']
        cx=min(int((x1+x2)/2/W*(gw-1)),gw-1); cy=min(int((y1+y2)/2/H*(gh-1)),gh-1)
        density[cy,cx]+=1
    sns.heatmap(density,ax=ax4,cmap='YlOrRd',annot=True,fmt='.0f',cbar=True)
    ax4.set_title('Player Density Grid',fontsize=12,fontweight='bold')
    ax5=fig.add_subplot(2,3,5)
    if bg is not None: ax5.imshow(cv2.cvtColor(bg,cv2.COLOR_BGR2RGB),alpha=0.4)
    tids=player_df['track_id'].unique()[:5]; tc=plt.cm.Set1(np.linspace(0,1,max(len(tids),1)))
    for tid,color in zip(tids,tc):
        traj=player_df[player_df['track_id']==tid].sort_values('frame')
        xs=[(r[0]+r[2])/2 for r in traj['xyxy']]; ys=[(r[1]+r[3])/2 for r in traj['xyxy']]
        ax5.plot(xs,ys,color=color,lw=2,label=f'Player #{tid}',alpha=0.8)
        if xs: ax5.scatter(xs[0],ys[0],color=color,s=80,zorder=5)
    ax5.set_xlim(0,W); ax5.set_ylim(H,0)
    ax5.set_title('Player Trajectories (first 5)',fontsize=12,fontweight='bold'); ax5.legend(fontsize=8); ax5.axis('off')
    ax6=fig.add_subplot(2,3,6)
    fr_range=range(int(df['frame'].max())+1); active=[len(player_df[player_df['frame']==f]) for f in fr_range]
    ax6.fill_between(fr_range,active,alpha=0.5,color='#2196F3')
    ax6.plot(fr_range,active,color='#1565C0',lw=1.5)
    ax6.set_title('Active Players Over Time',fontsize=12,fontweight='bold')
    plt.suptitle('Spatial Analytics Dashboard',fontsize=15)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR,'spatial_analytics.png'),dpi=130,bbox_inches='tight')
    plt.show()
    print('Spatial analytics saved!')


---
## 💾 CELL 18 — Auto-Save Everything to Google Drive

**Creates:** `Big_Vision_TOTAL_BACKUP/Basketball_Project_Final/`

**⚠️ Optional:** Change the `SAVE_DIR` folder name.

In [ ]:
import shutil
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else: print('Drive already mounted.')
# ════════════════════════════════════════════════════════
# ⚠️  OPTIONAL: change folder name
SAVE_DIR='/content/drive/MyDrive/Big_Vision_TOTAL_BACKUP/Basketball_Project_Final/'
# ════════════════════════════════════════════════════════
os.makedirs(SAVE_DIR, exist_ok=True); print(f'Saving to: {SAVE_DIR}')
files_to_save={
    '/content/runs/yolov11m_basketball/weights/best.pt':'best_model_yolov11m.pt',
    OUTPUT_VIDEO:'basketball_tracked.mp4',
    '/content/player_heatmap.jpg':'player_heatmap.jpg',
    '/content/spatial_analytics.png':'spatial_analytics.png',
    '/content/tracking_analytics.png':'tracking_analytics.png',
    '/content/learning_curves.png':'learning_curves.png',
    '/content/detection_metrics.png':'detection_metrics.png',
    '/content/lr_schedule.png':'lr_schedule.png',
    '/content/class_distribution.png':'class_distribution.png',
    '/content/sample_annotations.png':'sample_annotations.png',
    '/content/detection_test.png':'detection_test.png',
    '/content/merged_dataset/data.yaml':'data.yaml',
}
for src,dst in files_to_save.items():
    if os.path.exists(src): shutil.copy(src,SAVE_DIR+dst); print(f'  ✅ {dst}')
    else: print(f'  ⚠️  Not found: {dst}')
run_dir='/content/runs/yolov11m_basketball/'
if os.path.exists(run_dir):
    shutil.copytree(run_dir,SAVE_DIR+'training_run/',dirs_exist_ok=True)
    print('  ✅ training_run/ folder')
print(f'\nAll saved to: {SAVE_DIR}')

---
## 🎨 CELL 19 — Gradio Live Demo (Drag-and-Drop)

**Drag any basketball video → tracked output appears in browser.**

Accepts: uploaded file OR Google Drive path.

**Nothing to change.**

In [ ]:
import gradio as gr, tempfile
from ultralytics import YOLO
import supervision as sv

demo_model = YOLO(BEST_MODEL_PATH)

def track_video(video_file, drive_path, progress=gr.Progress()):
    input_path = (drive_path.strip() if drive_path and drive_path.strip() else video_file)
    if not input_path or not os.path.exists(str(input_path)):
        return None, 'Provide a valid video file or path'
    try:
        with tempfile.NamedTemporaryFile(suffix='.mp4',delete=False) as tmp: out=tmp.name
        vi=sv.VideoInfo.from_video_path(input_path)
        box_a=sv.BoxAnnotator(thickness=2); lbl_a=sv.LabelAnnotator(text_scale=0.5,text_padding=3)
        trc_a=sv.TraceAnnotator(thickness=2,trace_length=40); CNAMES={0:'Player',1:'Ball'}; plist=[]
        with sv.VideoSink(out,vi) as sink:
            for fi,frame in enumerate(sv.get_video_frames_generator(input_path)):
                r=demo_model.track(frame,tracker='bytetrack.yaml',persist=True,conf=0.40,iou=0.5,verbose=False)[0]
                dets=sv.Detections.from_ultralytics(r)
                if len(dets)>0:
                    keep=[]
                    for i in range(len(dets)):
                        x1,y1,x2,y2=dets.xyxy[i]; bw=x2-x1; bh=y2-y1
                        if int(dets.class_id[i])==0 and bh>0 and bh/max(bw,1)>5.5: continue
                        keep.append(i)
                    if keep: dets=dets[keep]
                labels=[f'#{int(dets.tracker_id[i]) if dets.tracker_id is not None else -1} {CNAMES.get(int(dets.class_id[i]),"?")} {float(dets.confidence[i]):.2f}' for i in range(len(dets))]
                ann=frame.copy(); ann=trc_a.annotate(ann,dets); ann=box_a.annotate(ann,dets); ann=lbl_a.annotate(ann,dets,labels)
                np_=sum(1 for c in dets.class_id if c==0); plist.append(np_)
                cv2.rectangle(ann,(0,0),(520,30),(0,0,0),-1)
                cv2.putText(ann,f'Frame:{fi} Players:{np_} ByteTrack',(5,20),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),1)
                sink.write_frame(ann); progress(fi/max(vi.total_frames,1))
        return out, f'Done! Frames:{vi.total_frames} Avg players:{np.mean(plist):.1f}'
    except Exception as e: return None, f'Error: {e}'

with gr.Blocks(title='Basketball Tracking',theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🏀 Basketball Player Detection & Tracking\nYOLOv11m + ByteTrack')
    with gr.Row():
        with gr.Column():
            vin=gr.Video(label='Upload Basketball Video (drag & drop)')
            drv=gr.Textbox(label='OR Google Drive Path',placeholder='/content/drive/MyDrive/game.mp4')
            btn=gr.Button('🏃 Track Video',variant='primary',size='lg')
            status=gr.Textbox(label='Status',lines=3,interactive=False)
        with gr.Column():
            vout=gr.Video(label='Tracked Output',autoplay=True)
    btn.click(track_video,[vin,drv],[vout,status])
    gr.Markdown('**Algorithm:** ByteTrack + YOLOv11m | **Classes:** Player, Ball')
print('Launching Gradio...'); demo.launch(share=True,debug=False)

---
## 📋 CELL 20 — Final Summary Report
**Copy into your submission documentation. Nothing to change.**

In [ ]:
print('='*65)
print('  BASKETBALL DETECTION & TRACKING — FINAL SUMMARY')
print('='*65)
print()
print('DETECTION:  YOLOv11m')
print('  Pre-trained: YES — COCO dataset (Microsoft, 118k imgs, 80 classes)')
print('  Source     : https://github.com/ultralytics/ultralytics')
print('  Approach   : Fine-tuning (starts from human-detection weights)')
print('  Why chosen : Real-time (30-100 FPS). 22% fewer params than YOLOv8.')
print('               Faster-RCNN=5-10FPS too slow. DETR needs too much data.')
print()
print('TRACKING:   ByteTrack')
print('  Pre-trained: YES — Kalman filter motion model (bytetrack.yaml)')
print('  Source     : ByteTrack (Zhang et al. 2022), Ultralytics built-in')
print('  Why chosen : Two-stage matching keeps low-conf detections alive.')
print('               Recovers occluded player tracks. MOTA=77.3%, 171 FPS.')
print('               SORT drops tracks at crossings. DeepSORT confused by')
print('               identical jerseys. ByteTrack handles both problems.')
print()
print('ACCURACY IMPROVEMENTS:')
print('  1. player+ball only — removed referee (was causing post false-detections)')
print('  2. Aspect ratio filter (height/width>5.5 = post shape, rejected)')
print('  3. Min area filter (<0.5% of image = noise, rejected)')
print('  4. 5 datasets merged (~5000+ images vs 3 datasets before)')
print('  5. Confidence threshold raised to 0.40')
print()
print('DATASETS: 5 Roboflow sources | Classes: player(0), ball(1)')
if 'EVAL_RESULTS' in dir():
    print()
    print('DETECTION RESULTS:')
    for k,v in EVAL_RESULTS.items(): print(f'  {k:12s}: {v:.4f}')
if 'tracking_data' in dir() and tracking_data:
    df_s=pd.DataFrame(tracking_data); tl=df_s.groupby('track_id').size()
    print()
    print('TRACKING RESULTS:')
    print(f'  Unique tracks   : {df_s["track_id"].nunique()}')
    print(f'  Avg track length: {tl.mean():.1f} frames')
    print(f'  Short tracks    : {(tl<5).sum()} (fewer=better)')
print()
print('OUTPUT FILES:')
print(f'  Model        : /content/runs/yolov11m_basketball/weights/best.pt')
print(f'  Tracked video: {OUTPUT_VIDEO}')
if 'SAVE_DIR' in dir(): print(f'  Drive backup : {SAVE_DIR}')
print()
print('SUBMISSION: Share Colab notebook link + Drive folder link')
print('='*65)